In [1]:
# Ruta del directorio actual
import os; ruta_actual = os.getcwd() 
# os.startfile(ruta_actual)

# CICLO DE ACOPLAMIENTO MT3D-USGS-WMA

# 1. Preparación del entorno, se genera "conc_i.m3d"

In [5]:
# Preparación del entorno
import os, sys
import numpy as np 
import flopy
import matplotlib.pyplot as plt
from flopy.utils.binaryfile import HeadFile, CellBudgetFile, UcnFile
from flopy.plot.styles import styles
import pandas as pd 

# Generando un arreglo sconc en conc_i.m3d
sconc = np.ones((1, 1, 31), dtype=float)
sconc[0,0,10]=200
# Escribe sconc en "conc_i.m3d"
np.savetxt("conc_i.m3d", sconc.reshape(-1, sconc.shape[-1]), fmt="%.1f")
print("Iniciamos sconc = conc_i.m3d ",sconc)

Iniciamos sconc = conc_i.m3d  [[[  1.   1.   1.   1.   1.   1.   1.   1.   1.   1. 200.   1.   1.   1.
     1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.
     1.   1.   1.]]]


# 2. Se construye RT.f90 para leer conc_i.m3d y generar conc_s.txt

In [8]:
%%writefile RT.f90
program duplicar_concentraciones
    implicit none
    real :: sconc(31)
    integer :: i
    character(len=100) :: archivo_entrada, archivo_salida

    archivo_entrada = "conc_i.m3d"
    archivo_salida = "conc_s.txt"

    ! Leer el arreglo desde el archivo
    open(unit=10, file=archivo_entrada, status='old', action='read')
    read(10, *) sconc
    close(10)

    print *, " Al ejecutar RT.exe:"
    print *, " Se lee conc_i.m3d que contiene:"
    write(*,'("sconc ",31F8.2)')sconc !Aquí hay que cambiar el formato de escritura

    ! print *, "sconc:", sconc
    ! Le resta 0.01
    sconc = sconc-0.01 !sconc_(n_i)=sconc_i-(suma(sconc_i/nf))

    ! Escribir el resultado en otro archivo
    open(unit=20, file=archivo_salida, status='replace', action='write')
    write(20, '(31F8.2)') sconc
    close(20)

    print *, "Y se genera conc_s.txt que contiene:"
    ! print *, "sconc_mod: ",sconc
    write(*,'("sconc ",31F8.2)')sconc !Aquí hay que cambiar el formato de escritura
end program duplicar_concentraciones

Writing RT.f90


# 3. Se ejecuta RT.exe Para leer conc_i.m3d y generar conc_s.txt

In [11]:
# COMPILAMOS Y EJECUTAMOS "RT"
!gfortran RT.f90 -o RT.exe
# AL EJECUTAR RT.EXE, Se lee conc_i.m3d y s genera conc_s.txt
!RT.exe

  Al ejecutar RT.exe:
  Se lee conc_i.m3d que contiene:
sconc     1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00  200.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00    1.00
 Y se genera conc_s.txt que contiene:
sconc     0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99  199.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99    0.99


# 4. Utilizando conc_s.txt, se construye BTN y se ejecuta MT3D-USGS nv_veces. La salida de MT3D-USGS se guarda en conc_i.m3d


In [14]:
# Esta función genera un ciclo con los puntos 1) y 2).
# Ejecuta GWF-MT3D-NVG.ipynb haciendo 1): 
# 1) Lee conc_s.txt para modificar transport.btn y ejecutar GWF-MT3D-NVG.ipynb para generar conc_i.m3d.
# 2) Ejecuta RT.exe que lee conc_i.m3d, lo modifica y genera conc_s.txt
# 3) Se repite el ciclo [1),2)]
#******************************************************************************************************
# 1. Importación de librerías
import numpy as np
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display

# 2. Rutas de archivos
entrada_txt = "conc_s.txt"
salida_m3d = "conc_i.m3d"
ejecutable = "RT.exe"

# 3. Slider para controlar nv
nv_slider = widgets.IntSlider(value=1, min=1, max=40, step=1, description="nv")
run_button = widgets.Button(description="Iniciar ciclo ")
output_area = widgets.Output()
contador_ejecuciones = 0

# 4. Función que ejecuta el ciclo
def ciclo_nv(b):
    global contador_ejecuciones
    output_area.clear_output()
    
    with output_area:
        contador_ejecuciones += 1
        print(f"🟢  Ejecución #{contador_ejecuciones}")        
        nv = nv_slider.value

        for i in range(nv):
            print(f"\n--- Iteración {i+1} de {nv} ---")
            
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
            # EJECUTAR: (GWF,MT3D) ó GWF-MT3D-NVG
            # Condición según contador de ejecuciones
            if contador_ejecuciones == 1 and i == 0:
                print("▶️✅ Primera ejecución: corriendo GWF.ipynb y MT3D.ipynb...")
                %run GWF.ipynb
                print(" 🟢🟢 Termina GWF, Inicia MT3D")
                %run MT3D.ipynb
            else:
                print("▶️✅ Solo corriendo MT3D.ipynb...")
                %run MT3D.ipynb
                #%run GWF-MT3D-NVG.ipynb

            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
                # Ejecutar RT.exe (o reemplazar por WMA)
            try:
                print(f"✅ Ejecutando {ejecutable} ...\n")
                proceso = subprocess.Popen([ejecutable], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

                for linea in proceso.stdout:
                    print(linea, end="")  # imprime la salida línea por línea en tiempo real

                proceso.wait()
                if proceso.returncode != 0:
                    print(f"❌ El ejecutable terminó con error. Código: {proceso.returncode}")
                    return  # o break, si estás dentro de un ciclo
            except Exception as e:
                print(f"❌ Error al ejecutar {ejecutable}: {e}")
                return  # o break

                
            print("✅ Construimos BTN, ejecutamos MT3D-USGS...\n")
            print("⏸️ Pausa de 1 segundo...\n")
            time.sleep(1)

# 5. Conectar botón
run_button.on_click(ciclo_nv)

# 6. Mostrar interfaz
display(nv_slider, run_button, output_area)

IntSlider(value=1, description='nv', max=40, min=1)

Button(description='Iniciar ciclo ', style=ButtonStyle())

Output()